# Classification Photo vs Non-Photo - Version Simplifiée

Notebook nettoyé et optimisé pour la classification binaire d'images.

## Exécution
**Exécutez toutes les cellules dans l'ordre (Run All)**

## 1. Imports et Configuration

In [ ]:
# Imports & Configuration (Generator pipeline)
import os
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

import tensorflow as tf
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras import layers, models
from sklearn.metrics import confusion_matrix, classification_report, precision_recall_fscore_support, roc_curve, auc
import pandas as pd

# Configuration
tf.keras.utils.set_random_seed(42)
IMG_SIZE = 128
BATCH_SIZE = 32
EPOCHS = 20

# Robust path resolution for DATA_DIR
# 1) If env var DATA_DIR exists, use it
# 2) Else try repo-relative ../data/raw (Windows-friendly)
# 3) Else fallback to workspace-style '/workspace/data/raw'
ENV_DATA_DIR = os.environ.get('DATA_DIR')
if ENV_DATA_DIR and os.path.isdir(ENV_DATA_DIR):
    DATA_DIR = ENV_DATA_DIR
else:
    # Resolve relative to this notebook's directory
    nb_dir = os.path.dirname(os.path.abspath(__file__)) if '__file__' in globals() else os.getcwd()
    candidate = os.path.abspath(os.path.join(nb_dir, '..', 'data', 'raw'))
    if os.path.isdir(candidate):
        DATA_DIR = candidate
    elif os.path.isdir('/workspace/data/raw'):
        DATA_DIR = '/workspace/data/raw'
    else:
        # Last resort: print a clear error and raise
        print('❌ DATA_DIR introuvable. Vérifiez le chemin de vos données:')
        print(' - Attendu: <repo>/data/raw avec sous-dossiers (Photos, non_photo)')
        print(' - Ou définissez la variable d\'environnement DATA_DIR vers ce dossier')
        raise FileNotFoundError("Aucun dossier de données valide trouvé pour DATA_DIR")

print(f"✅ TensorFlow {tf.__version__}")
print(f"✅ GPU disponible: {len(tf.config.list_physical_devices('GPU')) > 0}")
print(f"📁 DATA_DIR: {DATA_DIR}")
# Lister les sous-dossiers trouvés
try:
    subdirs = [d for d in os.listdir(DATA_DIR) if os.path.isdir(os.path.join(DATA_DIR, d))]
    print(f"📂 Sous-dossiers: {subdirs}")
except Exception as e:
    print(f"(info) Impossible de lister {DATA_DIR}: {e}")

## 2. Chargement des Données avec Équilibrage

### 2.b Équilibrage des classes (oversampling 50/50)

- Objectif: compenser le déséquilibre (≈ 39k non_photo vs 10k Photos)
- Méthode: créer un dataset d'entraînement équilibré par sur-échantillonnage (oversampling) → 50% non_photo / 50% Photos
- Avantages: simple, efficace, pas besoin de `class_weight`
- Remarque: l'oversampling se fait uniquement sur l'entraînement; la validation reste inchangée

Nous allons:
1) Séparer les exemples par classe
2) Répéter et mélanger chaque flux
3) Combiner avec `tf.data.Dataset.sample_from_datasets` (poids 0.5 / 0.5)
4) Rebatcher et précharger pour de bonnes perfs

In [ ]:
# 2. Générateurs ImageDataGenerator avec split validation
# On part d'un seul dossier DATA_DIR qui contient 2 sous-dossiers: ex. 'Photos' et 'non_photo'

# Generators de base
train_datagen = ImageDataGenerator(
    rescale=1./255,
    validation_split=0.2  # 20% validation
)

val_datagen = ImageDataGenerator(
    rescale=1./255,
    validation_split=0.2
)

# Générateur d'entraînement (subset training)
train_gen_raw = train_datagen.flow_from_directory(
    DATA_DIR,
    target_size=(IMG_SIZE, IMG_SIZE),
    batch_size=BATCH_SIZE,
    class_mode='binary',
    subset='training',
    shuffle=True,
    seed=42
)

# Générateur de validation (subset validation)
val_gen = val_datagen.flow_from_directory(
    DATA_DIR,
    target_size=(IMG_SIZE, IMG_SIZE),
    batch_size=BATCH_SIZE,
    class_mode='binary',
    subset='validation',
    shuffle=False
)

print("✅ Classes:", train_gen_raw.class_indices)
# Sauvegarde des noms de classes dans l'ordre des indices
idx_to_name = {v: k for k, v in train_gen_raw.class_indices.items()}
class_names = [idx_to_name[0], idx_to_name[1]]
print(f"➡️ class_names: {class_names}")

In [ ]:
# 3. Oversampling ciblé via deux générateurs
# On identifie la minoritaire et la majoritaire depuis class_indices
indices = train_gen_raw.class_indices
# On suppose 'Photos' et 'non_photo' dans DATA_DIR; adaptez si besoin
classes_sorted = sorted(indices.items(), key=lambda kv: kv[1])
class0, class1 = classes_sorted[0][0], classes_sorted[1][0]
print(f"Classes (ordre index): 0 -> {class0}, 1 -> {class1}")

# Générateur d'augmentation pour la minoritaire
augment_gen = ImageDataGenerator(
    rescale=1./255,
    rotation_range=20,
    width_shift_range=0.1,
    height_shift_range=0.1,
    zoom_range=0.2,
    horizontal_flip=True,
    fill_mode="nearest",
    validation_split=0.2
)

# Choisissons la minoritaire en se basant sur les effectifs du generator de validation (stable)
_tmp_val = val_datagen.flow_from_directory(
    DATA_DIR,
    target_size=(IMG_SIZE, IMG_SIZE),
    batch_size=BATCH_SIZE,
    class_mode='binary',
    subset='validation',
    shuffle=False
)
counts = np.bincount(_tmp_val.classes)
minority_index = int(np.argmin(counts))
majority_index = 1 - minority_index
minority_name = class0 if minority_index == 0 else class1
majority_name = class0 if majority_index == 0 else class1
print(f"Minoritaire: {minority_name} | Majoritaire: {majority_name}")

# IMPORTANT: utilisez class_mode=None pour éviter l'étiquette 0 forcée quand une seule classe est fournie
# Nous allons créer les labels nous-mêmes à partir de val_gen.class_indices pour rester cohérents

# Générateur pour la classe minoritaire (augmentation seulement, sans labels)
minority_aug_gen = augment_gen.flow_from_directory(
    DATA_DIR,
    classes=[minority_name],
    target_size=(IMG_SIZE, IMG_SIZE),
    batch_size=BATCH_SIZE,
    class_mode=None,  # <-- pas de labels auto
    subset='training',
    shuffle=True,
    seed=42
)

# Générateur pour la classe majoritaire (sans augmentation, sans labels)
majority_gen = train_datagen.flow_from_directory(
    DATA_DIR,
    classes=[majority_name],
    target_size=(IMG_SIZE, IMG_SIZE),
    batch_size=BATCH_SIZE,
    class_mode=None,  # <-- pas de labels auto
    subset='training',
    shuffle=True,
    seed=42
)

# Mapper nom de classe -> label 0/1 cohérent avec validation
label_map = val_gen.class_indices  # ex: {'Photos': 0, 'non_photo': 1} selon l'ordre
min_label_val = label_map[minority_name]
maj_label_val = label_map[majority_name]
print(f"Labels (validation mapping): {minority_name}={min_label_val}, {majority_name}={maj_label_val}")

# Générateur équilibré 50/50 avec labels corrects
def balanced_generator(min_gen, min_label, maj_gen, maj_label):
    while True:
        x_min = next(min_gen)  # (B, H, W, 3)
        x_maj = next(maj_gen)
        y_min = np.full((x_min.shape[0],), min_label, dtype=np.float32)
        y_maj = np.full((x_maj.shape[0],), maj_label, dtype=np.float32)
        x = np.concatenate([x_min, x_maj], axis=0)
        y = np.concatenate([y_min, y_maj], axis=0)
        # Mélange intra-batch pour éviter l'ordre fixe
        idx = np.random.permutation(len(y))
        yield x[idx], y[idx]

train_balanced_gen = balanced_generator(minority_aug_gen, min_label_val, majority_gen, maj_label_val)

# Steps/epoch: limité par la taille de la minoritaire pour rester équilibré
steps_per_epoch = min(len(minority_aug_gen), len(majority_gen))
print(f"steps_per_epoch (balanced): {steps_per_epoch}")

## 4. Construction du Modèle CNN

Architecture simple et efficace avec normalisation intégrée.

In [ ]:
# 4. Modèle avec régularisation (anti-overfitting)
reg = tf.keras.regularizers.l2(1e-4)

model = models.Sequential([
    layers.Conv2D(32, (3,3), activation='relu', input_shape=(IMG_SIZE, IMG_SIZE, 3), kernel_regularizer=reg),
    layers.BatchNormalization(),
    layers.MaxPooling2D(2,2),
    layers.SpatialDropout2D(0.1),

    layers.Conv2D(64, (3,3), activation='relu', kernel_regularizer=reg),
    layers.BatchNormalization(),
    layers.MaxPooling2D(2,2),
    layers.SpatialDropout2D(0.1),

    layers.Flatten(),
    layers.Dense(64, activation='relu', kernel_regularizer=reg),
    layers.Dropout(0.4),
    layers.Dense(1, activation='sigmoid')
])

model.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])
model.summary()

## 5. Vérification de l'équilibrage

On échantillonne quelques lots du générateur équilibré pour vérifier que les labels contiennent bien 0 et 1 en proportions ~50/50.

In [ ]:
# Sanity check: comptage des labels sur N batches équilibrés
N = 50
counts = {0: 0, 1: 0}
for i, (_, ys) in enumerate(train_balanced_gen):
    ys = ys.astype(int)
    counts[0] += int(np.sum(ys == 0))
    counts[1] += int(np.sum(ys == 1))
    if i + 1 >= N:
        break

total = counts[0] + counts[1]
ratio0 = counts[0] / total if total else 0
ratio1 = counts[1] / total if total else 0
print(f"Label 0: {counts[0]} ({ratio0:.2%}) | Label 1: {counts[1]} ({ratio1:.2%})")
print("✅ Équilibrage OK" if abs(ratio0 - 0.5) < 0.1 and abs(ratio1 - 0.5) < 0.1 else "⚠️ Vérifiez les générateurs")

## 6. Entraînement

In [ ]:
# 6. Entraînement avec générateur équilibré
checkpoint_path = os.path.abspath(os.path.join(os.path.dirname(DATA_DIR), '..', 'outputs', 'best_model.keras'))
os.makedirs(os.path.dirname(checkpoint_path), exist_ok=True)

callbacks = [
    tf.keras.callbacks.EarlyStopping(monitor='val_loss', patience=6, restore_best_weights=True),
    tf.keras.callbacks.ReduceLROnPlateau(monitor='val_loss', patience=3, factor=0.5, min_lr=1e-6),
    tf.keras.callbacks.ModelCheckpoint(checkpoint_path, monitor='val_loss', save_best_only=True)
]

print("\n🚀 Entraînement (balanced generator) avec anti-overfitting...\n")
history = model.fit(
    train_balanced_gen,
    steps_per_epoch=steps_per_epoch,
    validation_data=val_gen,
    epochs=EPOCHS,
    callbacks=callbacks
)
print("\n✅ Entraînement terminé! Meilleur modèle sauvegardé si améliorations.")

## 7. Courbes d'Apprentissage

In [ ]:
# 7. Courbes d'Apprentissage (balanced generator)
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

ax1.plot(history.history.get('loss', []), label='Train')
ax1.plot(history.history.get('val_loss', []), label='Validation')
ax1.set_title('Loss')
ax1.set_xlabel('Epoch')
ax1.legend()

ax2.plot(history.history.get('accuracy', []), label='Train')
ax2.plot(history.history.get('val_accuracy', []), label='Validation')
ax2.set_title('Accuracy')
ax2.set_xlabel('Epoch')
ax2.legend()

plt.tight_layout(); plt.show()

## 8. Évaluation

### 8.a Techniques anti-overfitting appliquées

- L2 (weight decay) sur toutes les couches denses/convolutions pour pénaliser les grands poids.
- BatchNormalization pour stabiliser l'entraînement.
- SpatialDropout2D dans les blocs conv pour décorréler les canaux (robuste à l'overfitting spatial).
- Dropout (0.4) avant la couche de sortie.
- EarlyStopping (val_loss) avec restore_best_weights=True.
- ReduceLROnPlateau pour diminuer le LR lorsque la validation stagne.
- ModelCheckpoint pour conserver automatiquement le meilleur modèle sur val_loss.

Ces choix visent à améliorer la généralisation sans trop complexifier le modèle.

### 8.b Calibration du seuil de décision

Les probabilités (sigmoïde) sont converties en classes avec un seuil par défaut de 0.5. Pour un dataset déséquilibré, on peut optimiser ce seuil sur l'ensemble de validation pour mieux équilibrer précision/recall (ex: maximiser le F1 macro, ou imposer un recall minimum pour "Photos").

In [ ]:
# 8. Évaluation (validation generator)
# Prédictions sur val_gen (shuffle=False)
y_pred_proba = model.predict(val_gen)
y_pred = (y_pred_proba >= 0.5).astype(int).flatten()

# y_true selon l'ordre du generator
y_true = val_gen.classes

# Noms de classes class_names déjà défini
a = {v: k for k, v in val_gen.class_indices.items()}
class_names = [a[0], a[1]]

# Matrice de confusion
cm = confusion_matrix(y_true, y_pred)
plt.figure(figsize=(8, 6))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', xticklabels=class_names, yticklabels=class_names)
plt.title('Matrice de Confusion')
plt.ylabel('Vrai')
plt.xlabel('Prédit')
plt.show()

# Rapport
print("\n" + classification_report(y_true, y_pred, target_names=class_names))

In [ ]:
# 8.b Rapport sous forme de tables
p, r, f1, s = precision_recall_fscore_support(y_true, y_pred, labels=[0,1], zero_division=0)
report_df = pd.DataFrame({
    'classe': class_names,
    'precision': p,
    'recall': r,
    'f1': f1,
    'support': s
})

macro = precision_recall_fscore_support(y_true, y_pred, average='macro', zero_division=0)
weighted = precision_recall_fscore_support(y_true, y_pred, average='weighted', zero_division=0)

summary_df = pd.DataFrame({
    'avg': ['macro', 'weighted'],
    'precision': [macro[0], weighted[0]],
    'recall': [macro[1], weighted[1]],
    'f1': [macro[2], weighted[2]]
})

display(report_df)
display(summary_df)

### 8.c Courbe ROC et AUC

La courbe ROC montre la performance du classifieur à différents seuils.

In [ ]:
# Courbe ROC
fpr, tpr, thresholds = roc_curve(y_true, y_pred_proba)
roc_auc = auc(fpr, tpr)

plt.figure(figsize=(8, 6))
plt.plot(fpr, tpr, color='darkorange', lw=2, label=f'ROC curve (AUC = {roc_auc:.3f})')
plt.plot([0, 1], [0, 1], color='navy', lw=2, linestyle='--', label='Chance')
plt.xlim([0.0, 1.0])
plt.ylim([0.0, 1.05])
plt.xlabel('False Positive Rate')
plt.ylabel('True Positive Rate')
plt.title('Receiver Operating Characteristic (ROC)')
plt.legend(loc="lower right")
plt.grid(alpha=0.3)
plt.show()

print(f"AUC Score: {roc_auc:.4f}")

## 9. Tests Finaux Optimisés

Section finale pour tester le modèle sur de vraies images des deux classes.

In [ ]:
# Fonction de test optimisée et réutilisable
def predict_image(image_path, threshold=0.5, show_image=True):
    """
    Prédit la classe d'une image avec option d'affichage.
    
    Args:
        image_path: Chemin vers l'image
        threshold: Seuil de décision (défaut: 0.5)
        show_image: Afficher l'image avec la prédiction (défaut: True)
    
    Returns:
        (proba, predicted_class, confidence)
    """
    from tensorflow.keras.preprocessing import image as keras_image
    
    # Charger et préparer l'image
    img = keras_image.load_img(image_path, target_size=(IMG_SIZE, IMG_SIZE))
    img_array = keras_image.img_to_array(img) / 255.0
    img_array = np.expand_dims(img_array, axis=0)
    
    # Prédiction
    proba = float(model.predict(img_array, verbose=0)[0][0])
    pred_idx = 1 if proba >= threshold else 0
    predicted_class = class_names[pred_idx]
    confidence = proba if pred_idx == 1 else (1.0 - proba)
    
    # Affichage optionnel
    if show_image:
        plt.figure(figsize=(5, 5))
        plt.imshow(img)
        plt.axis('off')
        color = 'green' if pred_idx == 1 else 'blue'
        plt.title(f"Prédit: {predicted_class}\nScore: {proba:.3f} | Confiance: {confidence:.2%}",
                  color=color, fontsize=12, weight='bold')
        plt.tight_layout()
        plt.show()
    
    return proba, predicted_class, confidence

### 9.a Test avec grille visuelle

Teste plusieurs images de chaque classe et les affiche dans une grille.

In [ ]:
# Test visuel sur grille d'images
import glob
from tensorflow.keras.preprocessing import image as keras_image

def test_visual_grid(n_per_class=6, threshold=0.5):
    """
    Affiche une grille d'images testées avec leurs prédictions.
    
    Args:
        n_per_class: Nombre d'images par classe à tester
        threshold: Seuil de décision
    """
    results = []
    images_data = []
    
    # Collecter les images de chaque classe
    for cls_name in class_names:
        cls_dir = os.path.join(DATA_DIR, cls_name)
        if not os.path.exists(cls_dir):
            print(f"⚠️  Dossier introuvable: {cls_dir}")
            continue
            
        samples = sorted(glob.glob(os.path.join(cls_dir, '*.jpg')))[:n_per_class]
        
        for img_path in samples:
            # Charger image
            img = keras_image.load_img(img_path, target_size=(IMG_SIZE, IMG_SIZE))
            img_array = keras_image.img_to_array(img) / 255.0
            
            # Prédire
            proba = float(model.predict(np.expand_dims(img_array, axis=0), verbose=0)[0][0])
            pred_idx = 1 if proba >= threshold else 0
            pred_name = class_names[pred_idx]
            
            # Stocker
            is_correct = (pred_name == cls_name)
            results.append({
                'attendue': cls_name,
                'predite': pred_name,
                'correct': is_correct,
                'score': proba,
                'path': img_path
            })
            images_data.append((img, cls_name, pred_name, proba, is_correct))
    
    # Affichage en grille
    n_images = len(images_data)
    cols = 4
    rows = (n_images + cols - 1) // cols
    
    fig, axes = plt.subplots(rows, cols, figsize=(16, rows * 4))
    axes = axes.flatten() if n_images > 1 else [axes]
    
    for idx, (img, true_cls, pred_cls, proba, correct) in enumerate(images_data):
        ax = axes[idx]
        ax.imshow(img)
        ax.axis('off')
        
        color = 'green' if correct else 'red'
        status = '✅' if correct else '❌'
        ax.set_title(f"{status} Vraie: {true_cls}\nPrédit: {pred_cls} ({proba:.3f})",
                    color=color, fontsize=10, weight='bold')
    
    # Masquer les axes vides
    for idx in range(n_images, len(axes)):
        axes[idx].axis('off')
    
    plt.tight_layout()
    plt.show()
    
    # Statistiques
    df = pd.DataFrame(results)
    accuracy = df['correct'].mean()
    
    print("=" * 70)
    print("📊 RÉSULTATS DU TEST")
    print("=" * 70)
    print(f"\n✅ Accuracy globale: {accuracy:.2%}")
    print(f"\n📋 Détail par classe:")
    
    for cls in class_names:
        cls_df = df[df['attendue'] == cls]
        if len(cls_df) > 0:
            cls_acc = cls_df['correct'].mean()
            print(f"   {cls:15s}: {cls_acc:.2%} ({cls_df['correct'].sum()}/{len(cls_df)} corrects)")
    
    print("=" * 70)
    
    return df

# Lancer le test
test_results = test_visual_grid(n_per_class=6, threshold=0.5)

### 9.b Test manuel sur une image spécifique

Pour tester une image de votre choix.

In [ ]:
# Test manuel sur une image spécifique
# Remplacez le chemin par votre image

# Exemple 1: Tester une photo
image_path_photo = os.path.join(DATA_DIR, "Photos", "photo_0001.jpg")
if os.path.exists(image_path_photo):
    print("🖼️  TEST: Photo")
    proba, pred, conf = predict_image(image_path_photo, threshold=0.5, show_image=True)
    print(f"   Score: {proba:.4f} | Prédiction: {pred} | Confiance: {conf:.2%}\n")

# Exemple 2: Tester une non-photo
image_path_non_photo = os.path.join(DATA_DIR, "non_photo", "000_1_1_sz1.jpg")
if os.path.exists(image_path_non_photo):
    print("🖼️  TEST: Non-Photo")
    proba, pred, conf = predict_image(image_path_non_photo, threshold=0.5, show_image=True)
    print(f"   Score: {proba:.4f} | Prédiction: {pred} | Confiance: {conf:.2%}\n")

# Pour tester votre propre image:
# proba, pred, conf = predict_image(r"C:\chemin\vers\votre\image.jpg", show_image=True)

## 10. Sauvegarde Finale

Le modèle entraîné est sauvegardé pour une utilisation future.

## 11. Diagnostic du Modèle

Si le modèle prédit mal (photos → non_photo), vérifions:
1. Le mapping classe → label (0/1)
2. Les prédictions brutes (sigmoid) sur quelques exemples de chaque classe
3. La distribution des prédictions sur validation

In [ ]:
# DIAGNOSTIC COMPLET
print("=" * 60)
print("📊 DIAGNOSTIC DU MODÈLE")
print("=" * 60)

# 1. Vérifier le mapping classe → label
print("\n1️⃣ MAPPING DES CLASSES")
print(f"   class_indices: {val_gen.class_indices}")
print(f"   class_names: {class_names}")
print(f"   ➡️ Label 0 = {class_names[0]}")
print(f"   ➡️ Label 1 = {class_names[1]}")

# 2. Tester sur quelques images de chaque classe
print("\n2️⃣ TEST SUR ÉCHANTILLONS")
import glob

for cls_name in class_names:
    cls_dir = os.path.join(DATA_DIR, cls_name)
    samples = sorted(glob.glob(os.path.join(cls_dir, '*.jpg')))[:3]
    
    print(f"\n   📂 Classe attendue: {cls_name}")
    for img_path in samples:
        from tensorflow.keras.preprocessing import image as keras_image
        img = keras_image.load_img(img_path, target_size=(IMG_SIZE, IMG_SIZE))
        x = keras_image.img_to_array(img) / 255.0
        x = np.expand_dims(x, axis=0)
        proba = float(model.predict(x, verbose=0)[0][0])
        pred_idx = 1 if proba >= 0.5 else 0
        pred_name = class_names[pred_idx]
        
        status = "✅" if pred_name == cls_name else "❌"
        print(f"      {status} {os.path.basename(img_path):30s} | Score: {proba:.4f} | Prédit: {pred_name}")

# 3. Distribution des scores sur validation
print("\n3️⃣ DISTRIBUTION DES SCORES (validation)")
y_val_proba = model.predict(val_gen, verbose=0).flatten()
y_val_true = val_gen.classes

for i, cls_name in enumerate(class_names):
    mask = (y_val_true == i)
    scores = y_val_proba[mask]
    mean_score = scores.mean()
    print(f"   {cls_name:15s} (label={i}) | Moyenne score: {mean_score:.4f} | Count: {mask.sum()}")

print("\n4️⃣ INTERPRÉTATION")
print("   - Si les Photos ont un score moyen < 0.5, le modèle les confond avec non_photo")
print("   - Vérifiez que label 1 correspond bien à 'Photos' (ou inversement)")
print("   - Si le mapping est inversé, il faut corriger la fonction de test ou réentraîner")
print("=" * 60)

### 11.b Vérification des labels d'entraînement

Vérifions que le générateur équilibré produit bien les bons labels pendant l'entraînement.

In [ ]:
# Inspecter un batch du générateur d'entraînement
print("🔍 INSPECTION DU GÉNÉRATEUR ÉQUILIBRÉ")
print("-" * 60)

# Recréer le générateur pour cette inspection
test_balanced_gen = balanced_generator(minority_aug_gen, min_label_val, majority_gen, maj_label_val)

# Prendre un batch
X_batch, y_batch = next(test_balanced_gen)

print(f"Forme du batch: {X_batch.shape}")
print(f"Labels uniques dans le batch: {np.unique(y_batch)}")
print(f"Comptage labels: {np.bincount(y_batch.astype(int))}")

# Mapper les labels aux noms de classes
print(f"\nMapping attendu (validation):")
print(f"  - {minority_name} → label {min_label_val}")
print(f"  - {majority_name} → label {maj_label_val}")

# Vérifier la cohérence
print(f"\nVérification:")
if min_label_val in y_batch and maj_label_val in y_batch:
    print(f"  ✅ Les deux labels ({min_label_val} et {maj_label_val}) sont présents")
else:
    print(f"  ❌ PROBLÈME: labels manquants dans le batch!")

print("-" * 60)

### 11.c Solution: Corriger l'interprétation ou réentraîner

Si le diagnostic montre un problème de mapping (ex: Photos a un score moyen < 0.5 alors que label=1), deux solutions:

**Solution A**: Inverser l'interprétation dans les fonctions de test (rapide)
**Solution B**: Réentraîner avec le bon mapping (propre)

In [ ]:
# 🔧 SOLUTIONS POUR CORRIGER LES PRÉDICTIONS
print("=" * 70)
print("💡 SOLUTIONS DISPONIBLES")
print("=" * 70)

# Analyser automatiquement le problème
y_val_proba = model.predict(val_gen, verbose=0).flatten()
y_val_true = val_gen.classes

# Détecter automatiquement si on a un problème de mapping
probleme_detecte = False
for i, cls_name in enumerate(class_names):
    mask = (y_val_true == i)
    scores = y_val_proba[mask]
    mean_score = scores.mean()
    
    # Si c'est "Photos" et le score moyen est bas (< 0.5), ou vice-versa
    if 'photo' in cls_name.lower() and 'non' not in cls_name.lower():
        if (i == 1 and mean_score < 0.5) or (i == 0 and mean_score > 0.5):
            probleme_detecte = True
            print(f"\n⚠️  PROBLÈME DÉTECTÉ!")
            print(f"   '{cls_name}' (label {i}) a un score moyen de {mean_score:.4f}")
            if i == 1:
                print(f"   → Le modèle prédit plutôt label 0 pour cette classe!")
            else:
                print(f"   → Le modèle prédit plutôt label 1 pour cette classe!")

if probleme_detecte:
    print("\n" + "=" * 70)
    print("? SOLUTION A: Correction rapide (inverser l'interprétation)")
    print("=" * 70)
    print("Exécutez la cellule suivante pour corriger immédiatement.")
    print("Avantage: Rapide, pas besoin de réentraîner")
    print("Inconvénient: Les fichiers sauvegardés gardent le mauvais mapping")
    
    print("\n" + "=" * 70)
    print("📋 SOLUTION B: Réentraînement propre")
    print("=" * 70)
    print("1. Vérifiez l'ordre alphabétique de vos dossiers:")
    print(f"   {sorted(os.listdir(DATA_DIR))}")
    print("2. Le premier alphabétiquement devient label 0, le second label 1")
    print("3. Pour corriger: renommez vos dossiers pour avoir le bon ordre")
    print("   Exemple: 'Photos' → '1_Photos' et 'non_photo' → '0_non_photo'")
    print("4. Puis relancez tout depuis la Section 2 (chargement des données)")
    print("\nAvantage: Solution définitive, mapping cohérent partout")
    print("Inconvénient: Nécessite de réentraîner (~10-20 min)")
else:
    print("\n✅ AUCUN PROBLÈME DÉTECTÉ")
    print("   Le mapping semble correct. Si vous voyez encore des erreurs,")
    print("   vérifiez les chemins d'images dans vos tests manuels.")

print("=" * 70)

### 11.d CORRECTION RAPIDE - À exécuter maintenant

Inversez le mapping class_names pour corriger les prédictions immédiatement.

In [ ]:
# 🔧 CORRECTION APPLIQUÉE AUTOMATIQUEMENT
print("=" * 70)
print("🔧 APPLICATION DE LA CORRECTION")
print("=" * 70)

# Sauvegarder l'ancien mapping
old_class_names = class_names.copy()
print(f"\n❌ ANCIEN mapping (incorrect):")
print(f"   Label 0 = {old_class_names[0]}")
print(f"   Label 1 = {old_class_names[1]}")

# INVERSER le mapping
class_names = [old_class_names[1], old_class_names[0]]

print(f"\n✅ NOUVEAU mapping (corrigé):")
print(f"   Label 0 = {class_names[0]}")
print(f"   Label 1 = {class_names[1]}")

print("\n💡 Explication:")
print("   Le modèle a appris Photos=0 et non_photo=1, mais donnait des scores")
print("   élevés pour tout (car il a appris à l'envers). En inversant class_names,")
print("   les fonctions de test vont maintenant interpréter correctement.")

print("\n📝 Prochaine étape:")
print("   Relancez les cellules de test (Sections 9, 10, 12) pour voir la correction.")
print("=" * 70)

# Vérification rapide
print("\n🔍 VÉRIFICATION RAPIDE (3 photos):")
import glob
cls_dir = os.path.join(DATA_DIR, "Photos")
samples = sorted(glob.glob(os.path.join(cls_dir, '*.jpg')))[:3]

for img_path in samples:
    from tensorflow.keras.preprocessing import image as keras_image
    img = keras_image.load_img(img_path, target_size=(IMG_SIZE, IMG_SIZE))
    x = keras_image.img_to_array(img) / 255.0
    x = np.expand_dims(x, axis=0)
    proba = float(model.predict(x, verbose=0)[0][0])
    pred_idx = 1 if proba >= 0.5 else 0
    pred_name = class_names[pred_idx]
    
    status = "✅" if "photo" in pred_name.lower() and "non" not in pred_name.lower() else "❌"
    print(f"   {status} {os.path.basename(img_path):30s} | Score: {proba:.4f} | Prédit: {pred_name}")

In [ ]:
# Sauvegarde du modèle final (vous pouvez aussi charger le checkpoint best_model.keras)
final_path = os.path.abspath(os.path.join(os.path.dirname(DATA_DIR), '..', 'outputs', 'photo_classifier_cleanV2.keras'))
os.makedirs(os.path.dirname(final_path), exist_ok=True)
model.save(final_path)
print(f"✅ Modèle sauvegardé → {final_path}")